# Across samples

A kernel body is one sample's. `x` is this sample's value, and nothing in the body
reaches another sample unless you say so. This page shows the three ways to say so:
reduce over every sample, read one sample's value, and store by key.

**Colab or Kaggle:** run this first cell (it installs the packages and downloads the helper files this notebook uses). Elsewhere it only checks the imports.

In [ ]:
# Colab setup, generated by python -m raptor_docs.nb colab_setup (do not edit).
import os

try:
    import google.colab  # noqa: F401
    HOSTED = True  # a Colab runtime
except ImportError:
    HOSTED = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # a Kaggle kernel

if HOSTED:
    import importlib.metadata, pathlib, re, shutil, subprocess, sys, urllib.request

    gpu = bool(shutil.which("nvidia-smi")) and subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0
    extra = ""
    if gpu:
        # the CUDA major the runtime already has: a CuPy that is installed, else torch's, else the driver's
        major = None
        for m in ("13", "12"):
            try:
                importlib.metadata.version(f"cupy-cuda{m}x")
                major = m
                break
            except importlib.metadata.PackageNotFoundError:
                pass
        if major is None:
            try:
                import torch
                major = "13" if str(torch.version.cuda).startswith("13") else None
            except ImportError:
                pass
        if major is None:
            smi = re.search(r"CUDA Version: (\d+)", subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
            major = "13" if smi and int(smi.group(1)) >= 13 else "12"
        extra = f"[cuda{major}]"
        print(f"GPU found: installing the {extra} extras to match this runtime.")
    pre = ["--pre"] if os.environ.get("RAPTOR_PRE") else []  # optional: test a release candidate
    ref = os.environ.get("RAPTOR_REF", "main")  # optional: fetch the helper files from this git ref
    if pre or "RAPTOR_REF" in os.environ:
        print(f"Pre-release test: installing with --pre, helper files from {ref}.")
    # our packages with --upgrade, so a runtime that kept an older release gets the current one
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pre, f"raptor-eagle{extra}", f"raptor-hawk{extra}"])
    if not gpu:
        print("Running on the CPU (no GPU in this runtime). For a GPU: " + ("Settings → Accelerator → GPU T4 x2." if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "Runtime → Change runtime type → T4 GPU."))

try:
    import eagle, hawk  # noqa: F401
except ImportError as err:
    raise ImportError(
        "eagle, hawk not found. Install with `pip install raptor-eagle raptor-hawk`. For a GPU, add [cuda12] (or [cuda13]) to raptor-eagle and raptor-hawk."
    ) from err

## Two views of a value

| You write | It is | Use it for |
|---|---|---|
| `x` | this sample's value (per-sample, "locked") | ordinary math |
| `hawk.all_samples(x)` | every sample's value (all-samples, "unlocked") | `.sum() .max() .min() .mean()`, `[i]`, `[mask]` |

`hawk.all_samples(x)` is never arithmetic and never stored whole. Compute per sample
first, `hawk.all_samples(x * 2)`, then index, mask or reduce it.

## Reduce over samples

Store a reduction to a `Mutable` output and that output is a **cell**: one value for
the whole run. A mask in square brackets reduces only the samples where it is true.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np

import eagle
import hawk
from hawk import Index, Matrix, Mutable, Scalar, Vector


@hawk.kernel
def stats(x: Scalar, total: Mutable[Scalar], mean: Mutable[Scalar], positive: Mutable[Scalar]):
    total = hawk.all_samples(x).sum()
    mean = hawk.all_samples(x).mean()
    positive = hawk.all_samples(x)[x > 0.0].sum()


@hawk.kernel
def peak(x: Scalar, hi: Mutable[Scalar]):
    hi = hawk.all_samples(x).max()


stats

<Kernel stats slots=8>

`eagle.deploy` builds a kernel for the host and the GPU and hands back every cell finished, as a one-element array:

In [3]:
x = np.random.default_rng(1).normal(size=100)
out = eagle.deploy(stats).run(x=x)
print({k: float(v[0]) for k, v in out.items()})
print("numpy:", x.sum(), x.mean(), x[x > 0].sum())
print("max:", float(eagle.deploy(peak).run(x=x)[0]), "numpy:", x.max())

{'mean': -0.0736121212729471, 'positive': 29.154644636394995, 'total': -7.361212127294709}
numpy: -7.361212127294709 -0.07361212127294708 29.154644636395
max: 2.1178387550510482 numpy: 2.1178387550510482


A kernel reduces with one operation: `.sum()` and `.mean()` together (as above), or
`.max()`, or `.min()`. Put a `.max()` in its own kernel, as `peak` does.

## Named axes

A value also has its own axes. `v.sum()` folds a vector's components and `m.sum(axis="rows")`
folds a matrix's rows; the axis names are `"components"`, `"rows"` and `"columns"`.
`"samples"` is the axis of `hawk.all_samples(...)`. Name it to keep the value's own shape, and
the cell has that shape too: a vector cell, a matrix cell. `.rows[i]`, `.columns[j]` and
`.components[k]` read one row, column or component by name.

In [4]:
@hawk.kernel
def frame(m: Matrix[2, 2], gram: Mutable[Matrix[2, 2]], top: Mutable[Vector[2]],
          colsum: Mutable[Vector[2]]):
    gram = hawk.all_samples(m).sum(axis="samples")                    # a 2x2 cell
    top = hawk.all_samples(m).rows[0].sum(axis="samples")             # one row of it
    colsum = hawk.all_samples(m.sum(axis="rows")).sum(axis="samples")  # own axis first


m = np.random.default_rng(2).normal(size=(4, 50))          # a matrix plane is (rows*columns, samples)
out = eagle.deploy(frame).run(m=m)
print("gram  :", [round(float(out[f"gram_{k}"][0]), 4) for k in range(4)], " numpy:", m.sum(1).round(4))
print("top   :", [round(float(out[f"top_{k}"][0]), 4) for k in range(2)], " numpy:", m[:2].sum(1).round(4))
print("colsum:", [round(float(out[f"colsum_{k}"][0]), 4) for k in range(2)],
      " numpy:", (m[:2] + m[2:]).sum(1).round(4))

gram  : [1.0977, -1.9397, 3.7114, -4.4244]  numpy: [ 1.0977 -1.9397  3.7114 -4.4244]
top   : [1.0977, -1.9397]  numpy: [ 1.0977 -1.9397]
colsum: [4.8091, -6.364]  numpy: [ 4.8091 -6.364 ]


A vector or matrix cell comes back as one array per entry, `gram_0 ... gram_3` (row-major).

## Read another sample, or store by key

`hawk.all_samples(expr)[i]` is `expr` as sample `i` computed it. `i` is an `Index` plane or
an integer value. The reverse direction is a store by key: `+=` adds each sample's value
into the bin the key names (many samples may share a bin), `=` stores it. A bool mask
stores where it is true.
A worked example with `sample_index`: [One sample, one lane](tutorials/09_per_sample_control).


In [5]:
@hawk.kernel
def reverse(x: Scalar, i: Index, y: Mutable[Scalar]):
    y = hawk.all_samples(x * 2.0)[i]


@hawk.kernel
def scatter(x: Scalar, key: Index, bins: Mutable[Scalar]):
    hawk.all_samples(bins)[key] += x


@hawk.kernel
def clip_high(x: Scalar, y: Mutable[Scalar]):
    y = x * 0.0
    hawk.all_samples(y)[x > 2.0] = x


x = np.arange(6.0)
i = np.array([5, 4, 3, 2, 1, 0], dtype=np.int64)
print("gather :", eagle.deploy(reverse).run(x=x, i=i))
print("scatter:", eagle.deploy(scatter).run(x=x, key=np.array([0, 1, 0, 1, 0, 1], dtype=np.int64)))
print("mask   :", eagle.deploy(clip_high).run(x=x))

gather : [10.  8.  6.  4.  2.  0.]
scatter: [6. 9. 0. 0. 0. 0.]
mask   : [0. 0. 0. 3. 4. 5.]


A by-key plane is one shared array indexed by key, so the same plane cannot also be written per sample.

## Cells and the reduction policy

`hawk.run` takes a cell as the output you bind: a `(1,)` array for a scalar cell, a `(columns,)`
array for a vector cell. It adds the partial sums on the host, and the result is bitwise equal
to eagle's `"ordered"` policy. Every eagle door (`eagle.deploy`, a graph, the JAX bridge)
hands back the same finished cell. To add up values yourself, use `eagle.reduce` with one of
three policies:

| `policy=` | What you get |
|---|---|
| `"compensated"` (default) | an accurate sum; on the GPU not bitwise the same run to run |
| `"ordered"` | the same bits every run, every thread count, CPU and GPU |
| `"auto"` | `"compensated"` in this release |

`max` and `min` are exact, so they ignore the policy.

In [6]:
import tempfile
import hawk.artifact

xs = np.random.default_rng(0).normal(size=1000)


@hawk.kernel
def energy(x: Scalar, total: Mutable[Scalar]):
    total = hawk.all_samples(x * x).sum()


work = tempfile.mkdtemp()
hawk.artifact.build(energy, work, targets=("host",))
cell = np.zeros(1)
hawk.run(hawk.load(work, "energy"), x=xs, total=cell)

ordered = np.zeros(1)
eagle.Graph().add_node(eagle.reduce(xs * xs, ordered, policy="ordered"), name="sum").build().launch()
print("hawk.run cell :", cell[0], " numpy:", (xs * xs).sum())
print("bitwise equal to eagle.reduce(policy='ordered'):", cell[0] == ordered[0])

hawk.run cell : 956.3530648422026  numpy: 956.3530648422026
bitwise equal to eagle.reduce(policy='ordered'): True


## Read a total in the same kernel

Reading a reduced value later in the body is allowed. The total exists only after every sample
has been added, so a read of it is a launch boundary: the kernel is built as **stages**, one
launch each, with the total passed between them as a cell. `hawk.run` and the eagle doors
(`Plan.run`, `bind`, the reverse, the torch, JAX and Warp bridges) run the stages in sequence.
Each stage is compiled and cached on its own, so editing one stage recompiles only that one.
A per-sample value a later stage needs is computed again there. A cell that only a later stage
reads is marked `internal` in the sidecar's `cells` and is not handed back to you.

In [7]:
import json, pathlib


@hawk.kernel
def share(x: Scalar, z: Mutable[Scalar]):
    z = x / hawk.all_samples(x).sum()      # reads the total: two stages


split_dir = tempfile.mkdtemp()
hawk.artifact.build_bundle([share], split_dir, targets=("host",))     # build() publishes one launch
sidecar = json.loads((pathlib.Path(split_dir) / "share.json").read_text())
print("stages:", [(s["kernel"], s["cells_in"], s["cells_out"]) for s in sidecar["stages"]])

xs = np.random.default_rng(0).uniform(1.0, 2.0, 10)
z = np.zeros(10)
hawk.run(hawk.load(split_dir, "share"), x=xs, z=z)
print("z sums to one:", np.isclose(z.sum(), 1.0), " matches numpy:", np.allclose(z, xs / xs.sum()))
try:
    hawk.run(hawk.load(split_dir, "share"), x=xs, z=z, count=5)
except Exception as exc:
    print("refused:", str(exc).split(": ", 1)[-1])

stages: [('share', [], ['cut0']), ('share_2', ['cut0'], [])]
z sums to one: True  matches numpy: True
refused: this kernel reads a value reduced over every sample, so it runs over every sample (no base= or count=)


`build_bundle` is the door for a split kernel; `hawk.artifact.build` publishes exactly one
launch and says so. `until_done` and `simulate` refuse a split kernel, because they step it
many times.

## Forms that are gone

Each retired form is refused with the line that replaces it:

In [8]:
from hawk.ir import HawkError


def refused(label, build):
    try:
        build()
    except Exception as exc:
        print(f"{label}: {str(exc).splitlines()[0].split(': ', 1)[-1]}")


def accum():
    hawk.Accum

def contribute():
    @hawk.kernel
    def k(x: Scalar, y: Mutable[Scalar]):
        y.contribute(x)

def add_at():
    @hawk.kernel
    def k(x: Scalar, key: Index, y: Mutable[Scalar]):
        y.add(x, at=key)

def own_sum():
    @hawk.kernel
    def k(x: Scalar, y: Mutable[Scalar]):
        y = x.sum()

def arithmetic():
    @hawk.kernel
    def k(x: Scalar, y: Mutable[Scalar]):
        y = hawk.all_samples(x) * 2.0


refused("Accum", accum)
refused("y.contribute(x)", contribute)
refused("y.add(x, at=key)", add_at)
refused("x.sum() of one sample", own_sum)
refused("all_samples(x) * 2", arithmetic)

Accum: declare Mutable[Scalar] and write hawk.all_samples(bins)[key] += term in the body
y.contribute(x): 'y': write y = hawk.all_samples(v).sum() (or .max()/.min()/.mean())
y.add(x, at=key): 'y': write hawk.all_samples(y)[key] += v (accumulate by key) or hawk.all_samples(y)[key] = v (store by key)
x.sum() of one sample: a rank-0 value is this sample's and has no axis to fold; the reduction over samples is hawk.all_samples(x).sum()
all_samples(x) * 2: 'hawk.all_samples(x)' is every sample; compute per sample first (hawk.all_samples(x * 2)), then index, mask or reduce it


| Retired | Write instead |
|---|---|
| `Accum[...]` plane, `y.add(v, at=key)` | `Mutable[Scalar]` and `hawk.all_samples(y)[key] += v` |
| `Reduce[...]` plane, `y.contribute(v)` | `Mutable[Scalar]` and `y = hawk.all_samples(v).sum()` |
| `y.write(v, at=key)` | `hawk.all_samples(y)[key] = v` |
| `x.batch`, `axis="samples"` on one sample's value | `hawk.all_samples(x)` |

## Limits

- A split kernel runs over every sample: `base=` and `count=` on `hawk.run` are refused. Doors that
  run single-stage kernels only (`until_done`, `simulate`, the step loops) refuse it by name:
  `'<kernel>' runs in N stages; <door> runs single-stage kernels in this release`. Use a plain
  kernel per step there.
- A reduction cannot sit inside a `for` loop, whether or not it is read back. Hoist it out;
  each cut is a separate pass over the samples.
- A matrix reduced over samples is not read back whole. Reduce the entries the kernel reads:
  `hawk.all_samples(m[i, j]).sum()`. A vector cell can be read back.
- One kernel reduces with one operation (sum or mean, or max, or min). Split the kernel.
- A by-key store is `+=` (a sum) or `=`; there is no by-key max or min yet.
- A by-key store takes a scalar plane: one `Mutable[Scalar]` per component (`u0`, `u1`, ...).
- A masked `.mean()` is refused because its count is a second cell: store
  `hawk.all_samples(x)[mask].sum()` and `hawk.all_samples(mask).sum()` and divide.
- The sample reached with `hawk.all_samples(expr)[i]` is recomputed from the inputs, so `expr`
  may not read a random draw (draws made inside a kernel are shown in [the ensemble example](examples/01_ensemble_drag_throughput)), a loop-carried value or a plane this launch writes.